# Clasificacion de urgencia con embeddings multilingues + Logistic Regression.
Mismos datos y mismo split que 02_baseline.py, para comparar de forma justa.

In [1]:
import os
import joblib
import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

load_dotenv()

MODELO_EMBEDDINGS = "paraphrase-multilingual-mpnet-base-v2"
VERSION = "v1"

/home/jorge/proyectos/sicpc-thesis/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### 1. Cargar datos (identico al baseline)

In [3]:
os.chdir("..")  # sube un nivel, a la raiz del proyecto
print(os.getcwd())
df_kaggle = pd.read_csv("data/processed/kaggle_es_baseline.csv")
df_kaggle["texto"] = df_kaggle["subject"].fillna("") + " " + df_kaggle["body"].fillna("")
df_kaggle = df_kaggle[["texto", "urgencia"]]
df_kaggle["fuente"] = "kaggle"

df_reales = pd.DataFrame(columns=["texto", "urgencia", "fuente"])
try:
    import mysql.connector

    conn = mysql.connector.connect(
        host="localhost", port=3306,
        user=os.getenv("DB_USER", "sJorge"), password=os.getenv("DB_PASSWORD"),
        database="sicpc",
    )
    df_reales = pd.read_sql(
        "SELECT asunto_anonimizado, cuerpo_anonimizado, urgencia_real "
        "FROM correo WHERE urgencia_real IS NOT NULL",
        conn,
    )
    conn.close()

    if len(df_reales) > 0:
        df_reales["texto"] = (
            df_reales["asunto_anonimizado"].fillna("") + " " +
            df_reales["cuerpo_anonimizado"].fillna("")
        )
        df_reales = df_reales.rename(columns={"urgencia_real": "urgencia"})
        df_reales = df_reales[["texto", "urgencia"]]
        df_reales["fuente"] = "real"
except Exception as e:
    print(f"No se pudieron cargar correos reales: {e}")

df = pd.concat([df_kaggle, df_reales], ignore_index=True)
df = df.dropna(subset=["texto", "urgencia"])
df = df[df["texto"].str.strip() != ""]

print(f"Total: {len(df)} filas")
print(df["fuente"].value_counts())


/home/jorge/proyectos/sicpc-thesis
Total: 962 filas
fuente
kaggle    812
real      150
Name: count, dtype: int64


/tmp/ipykernel_34745/3444991047.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_reales = pd.read_sql(


#### 2. Split (mismo que el baseline)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["texto"], df["urgencia"],
    test_size=0.2, random_state=42, stratify=df["urgencia"],
)

#### 3. Embeddings (el modelo NO se reentrena)

In [ ]:
print(f"\nCargando modelo de embeddings: {MODELO_EMBEDDINGS}")
encoder = SentenceTransformer(MODELO_EMBEDDINGS)

print("Generando embeddings de entrenamiento...")
X_train_emb = encoder.encode(
    X_train.tolist(), batch_size=32, show_progress_bar=True,
    normalize_embeddings=True,
)
print("Generando embeddings de prueba...")
X_test_emb = encoder.encode(
    X_test.tolist(), batch_size=32, show_progress_bar=True,
    normalize_embeddings=True,
)

#### 4. Clasificador

In [ ]:
modelo = LogisticRegression(max_iter=2000, class_weight="balanced")
modelo.fit(X_train_emb, y_train)

#### 5. Evaluacion

In [ ]:
y_pred = modelo.predict(X_test_emb)

print("\n" + "=" * 60)
print("REPORTE DE CLASIFICACION (embeddings)")
print("=" * 60)
print(classification_report(y_test, y_pred))

print("MATRIZ DE CONFUSION")
print(confusion_matrix(y_test, y_pred, labels=["alta", "media", "baja"]))
print("(orden de filas/columnas: alta, media, baja)")

# Metricas por fuente (Kaggle vs correos reales)
fuente_test = df.loc[X_test.index, "fuente"]
for f in ["kaggle", "real"]:
    mask = (fuente_test == f).values
    if mask.sum() > 0:
        print(f"\n--- Solo {f} (n={mask.sum()}) ---")
        print(classification_report(y_test[mask], y_pred[mask]))

#### 6. Guardar clasificador + nombre del encoder (van siempre juntos)

In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(
    {"clasificador": modelo, "encoder": MODELO_EMBEDDINGS},
    f"models/embeddings_{VERSION}.pkl",
)
print(f"\nGuardado en models/embeddings_{VERSION}.pkl")